# Niche model — raster → vector (03a)

Builds the national NiN raster from Light + Salinity + Substrate, vectorises and
generalises it (GRASS), and writes `…_generalized.gpkg`.

The vector overlays (tørrfall, observation substrate, land, attributes) live in
**`03b_nisjemodell_vector.ipynb`**, which reads the gpkg from here. The split lets
you iterate on the overlays without rerunning the expensive raster/GRASS step.

Pipeline (03a): Light + Salinity + Substrate → windowed raster classification
(3-bit key) + salinity usikkerhet band → vectorise + generalise (GRASS).

In [ ]:
from pathlib import Path

import numpy as np
import rasterio
from rasterio.vrt import WarpedVRT
from rasterio.windows import Window
from rasterio.enums import Resampling

import mnk
import mnk.attributes
import mnk.rules
import mnk.utils
import mnk.vectorize

CRS = "EPSG:25833"
EPSG = 25833
NODATA = 255

## Sources and paths

- **Light / reference grid**: national photic-zone raster (50 m, EPSG:25833, 1=euphotic, 0=aphotic, -1=nodata).
- **Brackish**: `Norfjords_160_surf_salt_mean_UTM33_brakkvann_50m_filled.tif` (thresholded and gap-filled, 1=brackish, 0=salt).
- **Substrate**: modelled 2-class substrate raster (2=hard).
- **Salinity**: raw HI surface salinity, for the salinity usikkerhet band.
- **Tørrfall, land, observed substrate**: used in the vector step (03b).

In [ ]:
# Local files live in the repository root (Jupyter runs from notebooks/, papermill from the root).
REPO = mnk.utils.repo_root()

photic_path = "gs://niva-geodata/MarintNaturKart/results/nisjedata-fotisk-sone-kd2020-2025_norge_2026_25833_padded.tif"
brakkvann_path = "gs://niva-geodata/MarintNaturKart/results/Norfjords_160_surf_salt_mean_UTM33_brakkvann_50m_filled.tif"

# 0=soft, 1=mixed, 2=hard, 255=nodata
substrat_raster_path = (
    "gs://niva-geodata/MarintNaturKart/results/nisjedata-substrat-xgbclassifier_norge_latest_25833_2class_padded.tif"
)

# Raw surface salinity for the salinity usikkerhet band.
salinity_path = "gs://niva-geodata/MarintNaturKart/input/hi/Norfjords_160_surf_salt_mean_UTM33.tif"

nin_raster_path = REPO / "nisjedata-modell_norge_2026_25833.tif"
# Intermediate rasters: plain class (1-12) and the sieved salinity band (kept for QA).
class_raster_path = REPO / "nisjedata-modell_norge_2026_25833_klasse.tif"
sal_band_path = REPO / "nisjedata-modell_norge_2026_25833_usikkerhet-salinitet.tif"
nin_vec_path = REPO / "nisjedata-modell_norge_2026_25833_generalized.gpkg"  # -> 03b leser denne

## NiN coding

Three binary dimensions (Light, Salinity, Substrate) are coded into a 3-bit key,
assuming **utenfor** (sublittoral) everywhere for now. Tørrfall is applied in the
vector step.

```
key = lys*4 + salinitet*2 + substrat
```

Each pixel also gets a **salinity** usikkerhet band (1 = lav, 2 = medium, 3 = høy)
from its signed distance to 18 PSU (`mnk.utils.salinity_margin` → `margin_band`,
thresholds `USIKKERHET_SAL`). A pixel on the wrong side of 18 PSU for its class, or
without salinity data (the class is then a gap-filled guess), is høy. The band is packed
into the raster value with `mnk.rules.encode_raster_class`, so polygons are split along
the salinity gradient and 03b reads the band straight from the class.

Light and substrate are not banded here: the depth model and the substrate
probabilities are too noisy per pixel, so 03b summarises them per polygon.

In [ ]:
# NiN integer (1-12) <-> axes and the 3-bit raster LUT live in mnk.rules
# (NIN_INT_AXES / NIN_INT_LUT3); 03b decodes the same integers back into axes.
NIN_INT_MAP = {k: mnk.rules.classify_nisje(*axes) for k, axes in mnk.rules.NIN_INT_AXES.items()}

# 3-bit LUT (tørrfall = outside):  key = lys*4 + salinitet*2 + substrat
#   0: Aphotic  Salt     Soft -> 8     1: Aphotic  Salt     Hard -> 12
#   2: Aphotic  Brackish Soft -> 6     3: Aphotic  Brackish Hard -> 10
#   4: Euphotic Salt     Soft -> 7     5: Euphotic Salt     Hard -> 11
#   6: Euphotic Brackish Soft -> 5     7: Euphotic Brackish Hard -> 9
LUT3 = np.array(mnk.rules.NIN_INT_LUT3, dtype=np.uint8)
print(LUT3)

## Windowed raster classification

For each row-stripe window:
1. read Light from the reference grid,
2. reproject brackish (WarpedVRT) onto the grid,
3. reproject the modelled substrate raster (WarpedVRT) onto the grid
   (2=fastbunn → hard, else soft),
4. compute the salinity band from the raw salinity (bilinear),
5. write the class (utenfor assumption) and the raw band to their own GeoTIFFs.

In [ ]:
BLOCK_ROWS = 1024
U = mnk.utils
sal_band_raw_path = sal_band_path.with_name(sal_band_path.stem + "_raw.tif")

with rasterio.open(photic_path) as photic:
    transform = photic.transform
    width, height = photic.width, photic.height
    photic_nodata = photic.nodata if photic.nodata is not None else -1

    profile = dict(
        driver="GTiff",
        height=height,
        width=width,
        count=1,
        dtype="uint8",
        crs=CRS,
        transform=transform,
        nodata=NODATA,
        compress="deflate",
        tiled=True,
        blockxsize=512,
        blockysize=512,
    )
    band_profile = {**profile, "nodata": 0}
    grid = dict(crs=CRS, transform=transform, width=width, height=height)

    brak_src = rasterio.open(brakkvann_path)
    brak_vrt = WarpedVRT(brak_src, resampling=Resampling.nearest, src_nodata=brak_src.nodata, nodata=255, **grid)
    sub_src = rasterio.open(substrat_raster_path)
    sub_vrt = WarpedVRT(sub_src, resampling=Resampling.nearest, src_nodata=sub_src.nodata, nodata=255, **grid)
    sal_src = rasterio.open(salinity_path)
    sal_vrt = WarpedVRT(sal_src, resampling=Resampling.bilinear, **grid)

    counts = np.zeros(13, dtype=np.int64)
    with rasterio.open(class_raster_path, "w", **profile) as cls_dst, \
            rasterio.open(sal_band_raw_path, "w", **band_profile) as band_dst:
        for row0 in range(0, height, BLOCK_ROWS):
            nrows = min(BLOCK_ROWS, height - row0)
            win = Window(0, row0, width, nrows)
            shp = (nrows, width)

            lys_raw = photic.read(1, window=win)
            lys_valid = lys_raw != photic_nodata
            lys = (lys_raw == 1).astype(np.uint8)  # 1=euphotic, 0=aphotic

            # Brackish only where VRT==1; coverage gaps count as salt
            brak = brak_vrt.read(1, window=win)
            salinitet = (brak == 1).astype(np.uint8)

            # 2=hard -> 1, 0/1 -> soft (0)
            sub_raw = sub_vrt.read(1, window=win)
            sub_valid = sub_raw != 255
            substrat = (sub_raw == 2).astype(np.uint8)

            valid = lys_valid & sub_valid
            nin = np.full(shp, NODATA, dtype=np.uint8)
            key = (lys[valid] * 4 + salinitet[valid] * 2 + substrat[valid]).astype(np.uint8)
            nin[valid] = LUT3[key]
            cls_dst.write(nin, 1, window=win)
            counts += np.bincount(nin[valid], minlength=13)

            sal = sal_vrt.read(1, window=win).astype("float32")
            if sal_vrt.nodata is not None:
                sal[sal == sal_vrt.nodata] = np.nan
            sal_band = U.margin_band(U.salinity_margin(sal, salinitet == 1), *U.USIKKERHET_SAL)
            band_dst.write(np.where(valid, sal_band, 0).astype(np.uint8), 1, window=win)
            del sal
            print(f"  rows {row0:>6}-{row0+nrows:<6} valid={int(valid.sum()):>12,}", end="\r")

    for ds in (brak_vrt, brak_src, sub_vrt, sub_src, sal_vrt, sal_src):
        ds.close()

print("\nNiN-fordeling (før tørrfall/observasjon/land):")
for i, (code_, name) in NIN_INT_MAP.items():
    if counts[i]:
        print(f"  {i:2d} {code_} {name}: {counts[i]:,}")

## Sieve the salinity band

`mnk.vectorize.sieve_raster` (GDAL sieve filter, streamed) merges band patches smaller
than `USIKKERHET_SIEVE_PX` (16 pixels = 4 ha) into their largest neighbour, so the band
does not leave single-pixel polygons. Only the band is sieved; the NiN classes are not
touched. On the 32VLQ test tile the salinity band added about 20 % polygons.

In [ ]:
mnk.vectorize.sieve_raster(sal_band_raw_path, sal_band_path, size=U.USIKKERHET_SIEVE_PX)
sal_band_raw_path.unlink()
print(f"Sieved salinity band -> {sal_band_path.name}")

## Combine class and band

Packs the class and the sieved salinity band into one raster value
(`mnk.rules.encode_raster_class`), which is what GRASS vectorises.

In [ ]:
band_counts = np.zeros(4, dtype=np.int64)
with rasterio.open(class_raster_path) as cls_src, \
        rasterio.open(sal_band_path) as band_src, \
        rasterio.open(nin_raster_path, "w", **profile) as dst:
    for row0 in range(0, height, BLOCK_ROWS):
        win = Window(0, row0, width, min(BLOCK_ROWS, height - row0))
        cls = cls_src.read(1, window=win)
        sal_band = band_src.read(1, window=win)
        valid = cls != NODATA
        nin = np.full(cls.shape, NODATA, dtype=np.uint8)
        nin[valid] = mnk.rules.encode_raster_class(cls[valid], sal_band[valid])
        dst.write(nin, 1, window=win)
        band_counts += np.bincount(sal_band[valid], minlength=4)

total = band_counts[1:].sum()
print("Usikkerhet salinitet: " + ", ".join(
    f"{k} {band_counts[i] / total:.1%}" for i, k in enumerate(mnk.attributes.USIKKERHET_KLASSER, start=1)
))

## Vectorise + generalise (GRASS)

Vectorises the NiN raster and removes the 50 m staircase artefacts in one
disk-streamed GRASS operation: `r.to.vect` → `v.generalize` (Douglas–Peucker,
25 m threshold). Topology is preserved so shared borders between NiN classes get no
gaps/overlaps. Memory-friendly because GRASS streams to disk.

In [ ]:
mnk.vectorize.with_grass(nin_raster_path, nin_vec_path)

## Next step

Run **`03b_nisjemodell_vector.ipynb`** — it reads `nin_vec_path`
(`…_generalized.gpkg`, class + salinity band) and applies tørrfall, observation
substrate, land, the light and substrate usikkerhet and the Naturkart attributes.